In [25]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

In [26]:
#pip install tensorflow

In [27]:

texts = [
    "I love this country",
    "This city is amazing",
    "Very good culture",
    "Excellent views",
    "I enjoyed the cafes",
    "The food was fantastic",
    "I hate this country",
    "This city is terrible",
    "Very boring view",
    "Worst place ever",
    "I disliked the country",
    "The city was awful"
]

In [28]:
labels = np.array([1, 1, 1, 1, 1, 1,
                   0, 0, 0, 0, 0, 0])

In [ ]:
#tokenize and pad sequences
# We set vocab_size to 1000 to keep only the top 1000 most common words in our dataset.
# The oov_token="<OOV>" argument tells the tokenizer to replace any word not in the top 1000 with a special token "<OOV>" (out-of-vocabulary).
# This helps handle words that may appear in new data but were not seen during training.
vocab_size = 1000
max_length = 6

# tokenize and pad sequences
tokenizer = Tokenizer(num_words=vocab_size, oov_token= "<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
X = pad_sequences(sequences, maxlen=max_length, padding="post")
print("word Index")
print(tokenizer.word_index)

print("\nInput Sequences:")
print(X)


word Index
{'<OOV>': 1, 'i': 2, 'this': 3, 'the': 4, 'country': 5, 'city': 6, 'is': 7, 'very': 8, 'was': 9, 'love': 10, 'amazing': 11, 'good': 12, 'culture': 13, 'excellent': 14, 'views': 15, 'enjoyed': 16, 'cafes': 17, 'food': 18, 'fantastic': 19, 'hate': 20, 'terrible': 21, 'boring': 22, 'view': 23, 'worst': 24, 'place': 25, 'ever': 26, 'disliked': 27, 'awful': 28}

Input Sequences:
[[ 2 10  3  5  0  0]
 [ 3  6  7 11  0  0]
 [ 8 12 13  0  0  0]
 [14 15  0  0  0  0]
 [ 2 16  4 17  0  0]
 [ 4 18  9 19  0  0]
 [ 2 20  3  5  0  0]
 [ 3  6  7 21  0  0]
 [ 8 22 23  0  0  0]
 [24 25 26  0  0  0]
 [ 2 27  4  5  0  0]
 [ 4  6  9 28  0  0]]


In [ ]:
# positional embedding in transformers
#creating custom layer that combine token and positional embedding
class TokenAndPositionEmbedding(layers.Layer): #1
    def __init__(self, max_length, vocab_size, embed_dim):
        super().__init__()
        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )

        self.position_embedding = layers.Embedding(
            input_dim=max_length,
            output_dim=embed_dim
        )

    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        token_emb = self.token_embedding(x)
        position_emb = self.position_embedding(positions)
        return token_emb + position_emb


In [ ]:
# 2. Create a custom Transformer block
# This block includes
# multi-head self-attention
# feed-forward network
# layer normalization and residual connections.
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])

        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()

    def call(self, inputs):
        # Self-attention (Query, Key, Value are calculated here using the same input)
        # The attention layer takes the input and computes
        # attention scores to capture relationships between different positions in the sequence.
        attention_output = self.attention(inputs, inputs)

        # Add + Normalize
        out1 = self.layernorm1(inputs + attention_output)

        # Feed-forward network
        ffn_output = self.ffn(out1)

        # Add + Normalize
        out2 = self.layernorm2(out1 + ffn_output)

        return out2

In [ ]:
# building the model
embed_dim = 16
num_heads = 2
ff_dim = 32
inputs = layers.Input(shape=(max_length,))

x = TokenAndPositionEmbedding(
    max_length=max_length,
    vocab_size=vocab_size,
    embed_dim=embed_dim
)(inputs)

x = TransformerBlock(
    embed_dim=embed_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
)(x)

x = layers.GlobalAveragePooling1D()(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)


In [33]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_5  │ (None, 6, 16)          │        16,096 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 6, 16)          │         3,296 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 16)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,409 (75.82 KB)

 Trainable params: 19,409 (75.82 KB)

 Non-trainable params: 0 (0.00 B)

In [34]:
model.fit(
    X,
    labels,
    epochs=30,
    batch_size=2,
    verbose=1
)

Epoch 1/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5833 - loss: 0.7220
Epoch 2/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7500 - loss: 0.6427 
Epoch 3/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6667 - loss: 0.5963 
Epoch 4/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9167 - loss: 0.5502 
Epoch 5/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9167 - loss: 0.5044 
Epoch 6/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9167 - loss: 0.4596 
Epoch 7/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.4854 
Epoch 8/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9167 - loss: 0.3846 
Epoch 9/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9167 - loss: 0.3332 
Epoch 10/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9167 - loss: 0.2902 
Epoch 11/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.2520 
Epoch 12/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.2293 
Ep

In [35]:
test_sentences = [ "I love this city",
                   "This country was awful"
                 ]
test_seq = tokenizer.texts_to_sequences(test_sentences)
test_pad = pad_sequences(test_seq, maxlen = max_length, padding="post")
predictions = model.predict(test_pad)

for sentence, prediction in zip(test_sentences, predictions):
    print(sentence, "->", prediction[0])
    if prediction[0] >0.5:
        print("Prediction: Positive")
    else:
        print("Prediction: Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
I love this city -> 0.94629514
Prediction: Positive
This country was awful -> 0.050023645
Prediction: Negative
